Load the raw data files.

In [3]:
import pandas as pd
import numpy as np

DATA_DIR = '/content'

df = pd.read_csv(f'{DATA_DIR}/company_master_wide.csv', low_memory=False)
cp = pd.read_csv(f'{DATA_DIR}/company_project.csv')
calvino = pd.read_csv(f'{DATA_DIR}/calvino_digital_intensity.csv')
pavitt = pd.read_csv(f'{DATA_DIR}/pavitt_taxonomy_nace2.csv')

print(f'Firms: {len(df):,}')
print(f'Grant participations: {len(cp):,}')

Firms: 14,139
Grant participations: 39,827


Parse NACE sector codes and map to NACE Rev. 2 section letters.

In [4]:
df['nace_4d'] = pd.to_numeric(df['NACE Rev. 2, core code (4 digits)'], errors='coerce')
df['nace_2d'] = (df['nace_4d'] // 100).astype('Int64')

NACE_SECTIONS = [
    (1,3,'A'), (5,9,'B'), (10,33,'C'), (35,35,'D'), (36,39,'E'), (41,43,'F'),
    (45,47,'G'), (49,53,'H'), (55,56,'I'), (58,63,'J'), (64,66,'K'), (68,68,'L'),
    (69,75,'M'), (77,82,'N'), (84,84,'O'), (85,85,'P'), (86,88,'Q'), (90,93,'R'),
    (94,96,'S'), (97,98,'T'), (99,99,'U')
]

def nace_to_section(code):
    if pd.isna(code):
        return 'Unknown'
    code = int(code)
    for lo, hi, section in NACE_SECTIONS:
        if lo <= code <= hi:
            return section
    return 'Unknown'

df['nace_section'] = df['nace_2d'].apply(nace_to_section)

print(f'Sections found: {df["nace_section"].nunique()}')
print(df['nace_section'].value_counts().head(10))

Sections found: 21
nace_section
C    4438
M    3893
J    1961
G     746
S     396
H     365
D     334
F     322
N     313
Q     251
Name: count, dtype: int64


Apply the four high-tech definitions.

In [5]:
# OECD R&D intensity (Galindo-Rueda & Verger, 2016)
oecd_ht_codes = {21, 26, 59, 60, 61, 62, 63, 72}
df['ht_oecd'] = df['nace_2d'].isin(oecd_ht_codes).astype(int)

# Calvino digital intensity (Calvino et al., 2018)
calvino_map = {}
for _, row in calvino.iterrows():
    isic = str(row['isic_rev4'])
    intensity = row['digital_intensity_2013_15']
    if '-' in isic:
        start, end = isic.split('-')
        for c in range(int(start), int(end) + 1):
            calvino_map[c] = intensity
    else:
        calvino_map[int(isic)] = intensity
df['ht_calvino'] = (df['nace_2d'].map(calvino_map) == 'High').astype(int)

# Pavitt taxonomy (Bogliacino & Pianta, 2016)
pavitt_map = {}
for _, row in pavitt.iterrows():
    nace = str(row['nace_rev2'])
    category = row['pavitt_category']
    if '-' in nace:
        start, end = nace.split('-')
        for c in range(int(start), int(end) + 1):
            pavitt_map[c] = category
    else:
        pavitt_map[int(nace)] = category

df['pavitt_category'] = df['nace_2d'].map(pavitt_map)
df['ht_pavitt'] = (df['pavitt_category'] == 'Science based').astype(int)
df['ht_pavitt_broad'] = df['pavitt_category'].isin(
    ['Science based', 'Specialised suppliers']).astype(int)

for col, label in [('ht_oecd', 'OECD'), ('ht_calvino', 'Calvino'),
                    ('ht_pavitt', 'Pavitt narrow'), ('ht_pavitt_broad', 'Pavitt broad')]:
    print(f'{label}: {df[col].mean():.1%} ({df[col].sum():,} firms)')

OECD: 30.2% (4,269 firms)
Calvino: 49.3% (6,972 firms)
Pavitt narrow: 32.1% (4,536 firms)
Pavitt broad: 61.7% (8,725 firms)


Classify funding instruments from CORDIS scheme codes.

In [6]:
def classify_instrument(scheme):
    s = str(scheme).upper()
    if any(x in s for x in ['SME-1', 'SME-2', 'EIC', 'FTI']):
        return 'SME'
    if any(x in s for x in ['ERC', 'MSCA']):
        return 'Early-stage'
    if any(x in s for x in ['RIA', 'CSA', 'ECSEL', 'JTI', 'BBI', 'IMI', 'FCH', 'SESAR', 'S2R', 'CLEAN SKY']):
        return 'Collaborative'
    if s == 'IA' or s.startswith('IA-') or '-IA' in s:
        return 'Collaborative'
    return 'Other'

cp['ecContribution_cordis'] = pd.to_numeric(cp['ecContribution_cordis'], errors='coerce')
cp['instrument'] = cp['project_fundingScheme'].apply(classify_instrument)
cp['startDate'] = pd.to_datetime(cp['startDate'], errors='coerce')
cp['grant_year'] = cp['startDate'].dt.year

print(cp['instrument'].value_counts())

instrument
Collaborative    33359
SME               2683
Early-stage       2184
Other             1601
Name: count, dtype: int64


Assign primary instrument per firm based on the largest share of EC funding.

In [7]:
firm_by_inst = cp.groupby(['organisationID', 'instrument'])['ecContribution_cordis'].sum().reset_index()
primary = (firm_by_inst
           .sort_values('ecContribution_cordis', ascending=False)
           .drop_duplicates('organisationID'))
primary = primary[['organisationID', 'instrument']].rename(
    columns={'instrument': 'primary_instrument'})

df = df.merge(primary, on='organisationID', how='left')
print(df['primary_instrument'].value_counts())

primary_instrument
Collaborative    10926
SME               1846
Other              686
Early-stage        681
Name: count, dtype: int64


Compute the predetermined treatment: total EC contribution from grants starting at or before the first grant year.

In [8]:
first_grant = cp.groupby('organisationID')['grant_year'].min().reset_index()
first_grant.columns = ['organisationID', 'first_grant_year_cp']

cp_with_first = cp.merge(first_grant, on='organisationID', how='left')
cp_pre = cp_with_first[cp_with_first['grant_year'] <= cp_with_first['first_grant_year_cp']]
firm_ec = cp_pre.groupby('organisationID')['ecContribution_cordis'].sum().reset_index()
firm_ec.columns = ['organisationID', 'ec_predetermined']

df = df.merge(firm_ec, on='organisationID', how='left')
df['log_ec'] = np.log(df['ec_predetermined'].where(df['ec_predetermined'] > 0))

n_unchanged = (df['ec_predetermined'] == df['total_ec_contribution']).sum()
n_reduced = (df['ec_predetermined'] < df['total_ec_contribution']).sum()
valid = df['ec_predetermined'].notna().sum()
print(f'Firms with predetermined EC: {valid:,}')
print(f'Unchanged (all grants in first year): {n_unchanged:,} ({n_unchanged/valid:.0%})')
print(f'Reduced (later grants excluded): {n_reduced:,} ({n_reduced/valid:.0%})')
print(f'Median ratio: {(df["ec_predetermined"] / df["total_ec_contribution"]).median():.3f}')

Firms with predetermined EC: 14,139
Unchanged (all grants in first year): 8,973 (63%)
Reduced (later grants excluded): 5,162 (37%)
Median ratio: 1.000


Classify firm independence status from the BvD independence indicator.

In [9]:
def classify_independence(val):
    v = str(val).strip().upper()
    if v.startswith('A'):
        return 'Independent'
    elif v.startswith('B'):
        return 'Partial'
    elif v.startswith(('C', 'D')):
        return 'Subsidiary'
    return 'Unknown'

df['independence'] = df['bvd_independence'].apply(classify_independence)
print(df['independence'].value_counts())

independence
Subsidiary     8329
Unknown        2981
Partial        1779
Independent    1050
Name: count, dtype: int64


Build financial time series matrices from the Orbis yearly columns.

In [10]:
def build_metric_matrix(df, prefix, n_years=30):
    matrix = np.full((len(df), n_years), np.nan)
    for j in range(n_years):
        col = f'{prefix}yr_minus_{j}'
        if j == 0 and f'{col}.1' in df.columns:
            col = f'{col}.1'
        if col in df.columns:
            matrix[:, j] = pd.to_numeric(df[col], errors='coerce').values
    return matrix

rev_matrix = build_metric_matrix(df, 'operating_revenue__turnover__')
emp_matrix = build_metric_matrix(df, 'number_of_employees__')
asset_matrix = build_metric_matrix(df, 'total_assets__')
pm_matrix = build_metric_matrix(df, 'profit_margin__')

for name, m in [('Revenue', rev_matrix), ('Employment', emp_matrix),
                ('Assets', asset_matrix), ('Profit margin', pm_matrix)]:
    pct = np.count_nonzero(~np.isnan(m)) / m.size * 100
    print(f'{name}: {pct:.1f}% non-missing')

Revenue: 23.6% non-missing
Employment: 25.6% non-missing
Assets: 28.7% non-missing
Profit margin: 21.1% non-missing


Compute grant-anchored timing, extract pre/post values, and build outcomes and controls.

In [11]:
df['last_year'] = pd.to_numeric(df['Last avail. year'], errors='coerce')
df['first_grant_year'] = pd.to_numeric(df['first_grant_year'], errors='coerce')
df['offset'] = df['last_year'] - df['first_grant_year']

offsets = df['offset'].values

def extract_post(matrix):
    return matrix[:, 0].copy()

def extract_pre_mean(matrix, offsets):
    n = len(offsets)
    result = np.full(n, np.nan)
    for i in range(n):
        o = offsets[i]
        if np.isnan(o):
            continue
        indices = [int(o) + 3, int(o) + 4, int(o) + 5]
        if all(0 <= idx < 30 for idx in indices):
            vals = [matrix[i, idx] for idx in indices if not np.isnan(matrix[i, idx])]
            if vals:
                result[i] = np.mean(vals)
    return result

# Outcomes: log-growth from pre-treatment average to post-treatment value
rev_post = extract_post(rev_matrix)
rev_pre = extract_pre_mean(rev_matrix, offsets)
df['rev_growth'] = np.where(
    (rev_post > 0) & (rev_pre > 0), np.log(rev_post) - np.log(rev_pre), np.nan)

emp_post = extract_post(emp_matrix)
emp_pre = extract_pre_mean(emp_matrix, offsets)
df['emp_growth'] = np.where(
    (emp_post > 0) & (emp_pre > 0), np.log(emp_post) - np.log(emp_pre), np.nan)

# Pre-treatment controls
df['log_rev_pre'] = np.where(rev_pre > 0, np.log(rev_pre), np.nan)
df['log_emp_pre'] = np.where(emp_pre > 0, np.log(emp_pre), np.nan)
assets_pre = extract_pre_mean(asset_matrix, offsets)
df['log_assets_pre'] = np.where(assets_pre > 0, np.log(assets_pre), np.nan)
df['pm_pre'] = extract_pre_mean(pm_matrix, offsets)

print(f'Offset range: {df["offset"].min():.0f} to {df["offset"].max():.0f}')
print(f'Revenue growth available: {df["rev_growth"].notna().sum():,}')
print(f'Employment growth available: {df["emp_growth"].notna().sum():,}')

/tmp/ipykernel_5380/4276340523.py:28: RuntimeWarning: divide by zero encountered in log
  (rev_post > 0) & (rev_pre > 0), np.log(rev_post) - np.log(rev_pre), np.nan)
/tmp/ipykernel_5380/4276340523.py:28: RuntimeWarning: invalid value encountered in log
  (rev_post > 0) & (rev_pre > 0), np.log(rev_post) - np.log(rev_pre), np.nan)
/tmp/ipykernel_5380/4276340523.py:28: RuntimeWarning: invalid value encountered in subtract
  (rev_post > 0) & (rev_pre > 0), np.log(rev_post) - np.log(rev_pre), np.nan)
/tmp/ipykernel_5380/4276340523.py:33: RuntimeWarning: divide by zero encountered in log
  (emp_post > 0) & (emp_pre > 0), np.log(emp_post) - np.log(emp_pre), np.nan)
/tmp/ipykernel_5380/4276340523.py:33: RuntimeWarning: invalid value encountered in subtract
  (emp_post > 0) & (emp_pre > 0), np.log(emp_post) - np.log(emp_pre), np.nan)
/tmp/ipykernel_5380/4276340523.py:36: RuntimeWarning: divide by zero encountered in log
  df['log_rev_pre'] = np.where(rev_pre > 0, np.log(rev_pre), np.nan)
/tmp/i

Offset range: -30 to 12
Revenue growth available: 4,663
Employment growth available: 4,688


Apply the three sample filters and report the attrition path.

In [12]:
# Filter 1: grant-anchored timing
timing_ok = (df['offset'] >= 1) & (df['offset'] + 5 <= 29) & df['offset'].notna()
df.loc[~timing_ok, ['rev_growth', 'emp_growth', 'log_rev_pre',
                      'log_emp_pre', 'log_assets_pre', 'pm_pre']] = np.nan
n_timing = timing_ok.sum()

# Filter 2: data depth (non-missing outcome and treatment)
has_rev = df['rev_growth'].notna() & df['log_ec'].notna()
has_emp = df['emp_growth'].notna() & df['log_ec'].notna()
n_depth_rev = has_rev.sum()
n_depth_emp = has_emp.sum()

# Filter 3: control completeness
controls_ok = (
    df[['log_rev_pre', 'log_emp_pre', 'log_assets_pre', 'pm_pre', 'offset']].notna().all(axis=1)
    & df['nace_2d'].notna()
    & df['Country ISO code'].notna()
    & df['first_grant_year'].notna()
)

keep_rev = has_rev & controls_ok
keep_emp = has_emp & controls_ok

print('Attrition path:')
print(f'  Full sample:           {len(df):>6,}')
print(f'  After timing filter:   {n_timing:>6,}')
print(f'  After data depth (rev):{n_depth_rev:>6,}')
print(f'  After data depth (emp):{n_depth_emp:>6,}')
print(f'  Final sample (rev):    {keep_rev.sum():>6,}')
print(f'  Final sample (emp):    {keep_emp.sum():>6,}')

Attrition path:
  Full sample:           14,139
  After timing filter:   13,437
  After data depth (rev): 4,394
  After data depth (emp): 4,407
  Final sample (rev):     3,750
  Final sample (emp):     3,497


Map countries to the 15 most common plus an Other category.

In [13]:
top_countries = df['Country ISO code'].value_counts().head(15).index.tolist()
df['country'] = df['Country ISO code'].where(
    df['Country ISO code'].isin(top_countries), 'Other')

print(df.loc[keep_rev, 'country'].value_counts())

country
ES       812
IT       761
FR       386
Other    369
DE       233
NO       193
SE       187
FI       183
BE       180
PT       150
PL        86
NL        65
CZ        49
DK        36
AT        33
IE        27
Name: count, dtype: int64


Save the analysis-ready dataset.

In [15]:
output_cols = [
    'organisationID', 'Country ISO code', 'country', 'nace_2d', 'nace_section',
    'ht_oecd', 'ht_calvino', 'ht_pavitt', 'ht_pavitt_broad',
    'primary_instrument', 'independence',
    'first_grant_year', 'last_year', 'offset',
    'ec_predetermined', 'log_ec',
    'rev_growth', 'emp_growth',
    'log_rev_pre', 'log_emp_pre', 'log_assets_pre', 'pm_pre',
]

out = df[output_cols].copy()
out.to_csv(f'{DATA_DIR}/analysis_ready.csv', index=False)

print(f'Saved {len(out):,} firms to analysis_ready.csv')
print(f'Columns: {len(output_cols)}')
print(f'Revenue sample: {keep_rev.sum():,}')
print(f'Employment sample: {keep_emp.sum():,}')

Saved 14,139 firms to analysis_ready.csv
Columns: 22
Revenue sample: 3,750
Employment sample: 3,497
